# RAG avançado na prática
## Quatro famílias de conserto, uma função trocada por vez

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

O pipeline da aula 8 não muda. O que muda é **uma função de contrato por vez**: `retrieve` na entrada (HyDE), `build_index` e `retrieve` (híbrido), `retrieve` na saída (reranking), um laço em volta de `retrieve` mais um `build_prompt` (CRAG). As mesmas 20 perguntas e o mesmo corpus da aula 8 passam pelas quatro variantes, e cada uma é comparada com a linha de base.

O notebook produz no fim uma tabela com cinco linhas (baseline e quatro variantes), três colunas de acerto (diretas, compostas, sem resposta) e o preço de cada linha em chamadas ao LLM. A pergunta que fecha a aula é qual técnica pagou o que custou.

## Como usar

Cada bloco tem um parágrafo de ideia, uma célula de demonstração, uma célula para você completar (com a solução escondida) e a célula que roda o harness nas 20 perguntas e grava o CSV. Rode na ordem; o índice do Chroma fica em disco e as células de harness reaproveitam o CSV se ele já existir. Tempo estimado, cerca de duas horas.

### Os oito problemas, as quatro famílias

O slide "Where vanilla RAG breaks" da aula 8 listou oito problemas. Aqui eles são agrupados pelo lugar do pipeline onde o conserto entra.

| Família | Problema da aula 8 | Técnica deste notebook | Também resolve (só citado) |
|---|---|---|---|
| **F1 · A pergunta** (antes de buscar) | Question / doc mismatch | HyDE | Step-Back, multi-query, decomposição |
| **F2 · A busca** (índice e retrieval) | One retriever · Arbitrary cuts · Isolated chunks | Híbrido BM25 + RRF | semantic chunking, parent-child, GraphRAG |
| **F3 · O contexto** (depois de buscar) | Lost in the middle | Reranking (cross-encoder) | compressão de contexto, reordenação |
| **F4 · O controle** (o sistema decide) | Blind trust · Same effort for all | CRAG | Adaptive RAG, Self-RAG, agentes (aula futura) |

*Amnesia* (memória de diálogo) e *Agentic RAG* ficam para a aula de agentes.

## Preparação (no próprio Colab)

Igual às aulas 7 e 8: no Colab o Ollama não vem instalado e a máquina é apagada ao fim da sessão, então a preparação é feita por células e repetida a cada sessão. Escolha **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**. Em CPU tudo funciona, só mais devagar (o gerador de 3B leva uns 20 s por resposta; o modelo de embedding é rápido nos dois casos).

| Passo | O que faz | Quanto tempo |
|---|---|---|
| 1 · Instalar | Instala o Ollama na máquina virtual. | ~1 min |
| 2 · Subir | Sobe o servidor em segundo plano. | segundos |
| 3 · Baixar | Baixa o gerador e o modelo de embedding. | ~2 min |
| 4 · Corpus | Baixa os 12 documentos do manual e as 20 perguntas (os mesmos da aula 8). | segundos |

Se você estiver rodando na sua máquina com o Ollama já instalado, pule os passos 1 a 3 e rode só o 4 e a instalação dos pacotes.

In [ ]:
# Passo 1 · Instala o Ollama na máquina virtual do Colab.
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 3
!ollama --version

In [ ]:
# Passo 2 · Sobe o servidor em segundo plano e espera até ele responder.
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
print("servidor no ar:", server_is_up(), requests.get(f"{OLLAMA_URL}/api/version").json())

In [ ]:
# Passo 3 · Baixa os modelos. O gerador de 3B faz três papéis novos nesta aula: escritor (HyDE), avaliador e reescritor (CRAG).
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:0.5b
!ollama pull nomic-embed-text
!ollama list

In [ ]:
# Passo 4 · Corpus e perguntas. Os mesmos arquivos da aula 8 (pasta week06 do repositório). Baixa se não existirem localmente.
import os, requests
RAW = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week06"
os.makedirs("corpus", exist_ok=True)
for f in ['car_specs.md', 'comms_protocol.md', 'drivers.md', 'fuel_and_energy.md', 'garage_and_pitlane.md', 'handbook_procedures.md', 'handbook_tyres.md', 'logistics.md', 'procedure_codes.md', 'radio_glossary.md', 'sponsor_obligations.md', 'strategy_playbook.md']:
    p = os.path.join("corpus", f)
    if not os.path.exists(p):
        open(p, "w", encoding="utf-8").write(requests.get(f"{RAW}/corpus/{f}").text)
if not os.path.exists("questions.json"):
    open("questions.json", "w", encoding="utf-8").write(requests.get(f"{RAW}/questions.json").text)
print(sorted(os.listdir("corpus")))

In [ ]:
# Instala o que falta. sentence-transformers traz o cross-encoder do Bloco 3 (uns 90 MB na primeira carga).
!pip -q install openai chromadb requests pandas matplotlib scikit-learn sentence-transformers

In [ ]:
# Núcleo do RAG clássico da aula 8, copiado sem alterações (só o diretório do Chroma mudou). Nada aqui é novo.
import os, re, glob, json, time, requests
from openai import OpenAI
import chromadb

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"            # o gerador (o Léo). Troque por "qwen2.5:0.5b" se a máquina sofrer.
EMB_MODEL  = "nomic-embed-text"      # o modelo de embedding. Tem de ser O MESMO na indexação e na consulta.
CORPUS_DIR = "corpus"
CHROMA_DIR = "chroma_lesson09"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ---- tokens. Aqui 1 token = 1 palavra (com o espaço que a precede), para o notebook não depender de nada.
# Tokenizadores reais (BPE) produzem cerca de 30% mais tokens que palavras. A proporção entre os tamanhos é o que importa.
def tokenize(text):   return re.findall(r"\s*\S+", text)
def detokenize(toks): return "".join(toks).strip()
def count_tokens(text): return len(tokenize(text))

# ---- Load
def load_corpus(folder=CORPUS_DIR):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*.md"))):
        docs.append({"doc": os.path.basename(path), "text": open(path, encoding="utf-8").read()})
    return docs

# ---- Chunk (tamanho fixo com overlap, em tokens)
def chunk_fixed(text, chunk_size=512, overlap=51):
    toks, step, out = tokenize(text), chunk_size - overlap, []
    for start in range(0, max(len(toks), 1), step):
        out.append(detokenize(toks[start:start + chunk_size]))
        if start + chunk_size >= len(toks):
            break
    return out

def make_chunks(docs, chunk_size=512, overlap=51):
    """Uma linha por chunk: id, texto e metadados (documento, posição, última seção vista)."""
    rows = []
    for d in docs:
        carried = "-"                                          # última seção vista no chunk anterior
        for i, c in enumerate(chunk_fixed(d["text"], chunk_size, overlap)):
            heads = [(m.start(), m.group(1)[:80]) for m in re.finditer(r"^#+ (.+)$", c, re.M)]
            first_half = [h for pos, h in heads if pos < len(c) / 2]
            section = first_half[-1] if first_half else carried   # a seção que domina o chunk
            if heads:
                carried = heads[-1][1]
            rows.append({"id": f"{d['doc']}#{i}", "text": c,
                         "doc": d["doc"], "chunk": i, "section": section})
    return rows

GROUNDING = ("You are the pit-wall assistant of Aurora Racing. Answer using ONLY the context below. "
             "If the context does not contain the answer, say exactly: \"The handbook does not cover this.\" "
             "Every fact you state must carry its source in square brackets, copied from the context labels, "
             "like [handbook_tyres.md · 2.2 Reference pressures]. "
             "End your answer with one line in exactly this format:\n"
             "Sources: [file · section], [file · section]")

def sources_used(hits):
    """A lista de fontes vem da recuperação, não do modelo. É o que se mostra ao usuário."""
    seen, out = set(), []
    for h in hits:
        key = f"{h['doc']} · {h['section']}"
        if key not in seen:
            seen.add(key); out.append(key)
    return out

# ---- Embed (Ollama /api/embed devolve uma lista de vetores)
def embed(texts, model=EMB_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": texts})
    r.raise_for_status()
    return r.json()["embeddings"]

# ---- Store (Chroma persistente, distância = 1 - cosseno)
def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)
    if rebuild:
        try: store.delete_collection(name)
        except Exception: pass
    col = store.get_or_create_collection(name, metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL})
    if col.count() == len(rows):
        return col                                   # já indexado; não paga o embedding de novo
    for i in range(0, len(rows), batch):
        part = rows[i:i + batch]
        col.upsert(ids=[r["id"] for r in part], documents=[r["text"] for r in part],
                   embeddings=embed([r["text"] for r in part]),
                   metadatas=[{"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]} for r in part])
    return col

# ---- Retrieve
def retrieve(col, question, k=4):
    assert col.metadata.get("embedding_model") == EMB_MODEL, "índice e consulta com modelos de embedding diferentes"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

# ---- Augment
GROUNDING = ("You are the pit-wall assistant of Aurora Racing. Answer using ONLY the context below. "
             "If the context does not contain the answer, say exactly: \"The handbook does not cover this.\" "
             "Cite the source of each fact in square brackets, like [handbook_tyres.md].")
FREE = "You are the pit-wall assistant of Aurora Racing. Answer the question."

def build_prompt(question, hits, grounded=True):
    context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
    system = GROUNDING if grounded else FREE
    user = f"Context:\n{context}\n\nQuestion: {question}" if hits else f"Question: {question}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

# ---- Generate
def generate(messages, model=GEN_MODEL, temperature=0.0, max_tokens=300):
    t0 = time.time()
    resp = client.chat.completions.create(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return resp.choices[0].message.content.strip(), time.time() - t0

def warm_up():
    """Carrega os dois modelos na memória e pede ao Ollama para mantê-los lá (keep_alive=-1). Evita pagar a carga na primeira pergunta."""
    requests.post(f"{OLLAMA_URL}/api/embed", json={"model": EMB_MODEL, "input": "warm up", "keep_alive": -1}).raise_for_status()
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": GEN_MODEL, "prompt": "hi", "keep_alive": -1, "stream": False,
                                                       "options": {"num_predict": 1}}).raise_for_status()
    for m in requests.get(f"{OLLAMA_URL}/api/ps").json().get("models", []):
        where = "GPU" if m.get("size_vram", 0) >= m.get("size", 1) * 0.9 else ("parcial GPU" if m.get("size_vram", 0) else "CPU")
        print(f"{m['name']:<24} carregado em {where}")

def show(hits):
    for i, h in enumerate(hits, 1):
        print(f"[{i}] {h['doc']} · {h['section']}  (distance {h['distance']:.3f})")
        print("    " + h["text"][:160].replace("\n", " ") + " ...")


In [ ]:
# Cronômetro automático. A partir daqui toda célula imprime o seu tempo de execução (⏱) e o registra em CELL_TIMES.
import time
from IPython import get_ipython

CELL_TIMES = []
_ip = get_ipython()

def _pre_run(info):
    _ip._t0 = time.perf_counter()

def _post_run(result):
    dt = time.perf_counter() - getattr(_ip, "_t0", time.perf_counter())
    first_line = (result.info.raw_cell.strip().splitlines() or [""])[0][:70]
    CELL_TIMES.append({"cell": first_line, "seconds": round(dt, 2)})
    print(f"⏱ {dt:.1f} s")

if not getattr(_ip, "_timer_installed", False):          # evita registrar duas vezes se a célula for rerodada
    _ip._t0 = time.perf_counter()
    _ip.events.register("pre_run_cell", _pre_run)
    _ip.events.register("post_run_cell", _post_run)
    _ip._timer_installed = True
print("cronômetro ligado")


## O núcleo da aula 9

A célula abaixo é o notebook inteiro em código. Leia os comentários de cabeçalho de cada família antes de rodar os blocos; cada bloco vai usar um pedaço dela. Há só duas mudanças globais: `generate` ganha um embrulho que conta chamadas ao LLM em `LLM_CALLS` (a coluna de custo da tabela final) e `retrieve` passa a devolver também o `id` do chunk (a moeda da fusão e do reranking). Todo o resto da aula 8 fica intacto.

In [ ]:
# Núcleo da aula 9. Quatro variantes do RAG, cada uma trocando UMA função de contrato da aula 8.
#   Família 1 (a pergunta)   · HyDE            → troca retrieve na entrada (o que é embutido)
#   Família 2 (a busca)      · Hybrid + RRF    → troca build_index e retrieve (dois índices, uma fusão)
#   Família 3 (o contexto)   · Reranking       → troca retrieve na saída (recupera mais, reordena, corta)
#   Família 4 (o controle)   · CRAG            → envolve retrieve num laço e troca build_prompt quando não há evidência
import math, re, time
from collections import Counter, defaultdict
import pandas as pd

# ---------------------------------------------------------------------------------------------------
# Duas mudanças globais, e só duas: generate passa a contar chamadas (a coluna de custo da tabela final)
# e retrieve passa a devolver o id do chunk (a moeda da fusão e do reranking).
# ---------------------------------------------------------------------------------------------------
LLM_CALLS = 0
_generate_lesson08 = generate                      # a função da aula 8, guardada antes de ser embrulhada

def generate(messages, model=GEN_MODEL, temperature=0.0):
    global LLM_CALLS
    LLM_CALLS += 1
    return _generate_lesson08(messages, model=model, temperature=temperature)

# ---------------------------------------------------------------------------------------------------
# Ajuda comum: o Chroma devolve ids; a aula 8 descartava. Aqui os ids são a moeda da fusão.
# ---------------------------------------------------------------------------------------------------
ROWS_BY_ID = {}                                    # preenchido por register_rows(rows)

def retrieve(col, question, k=4):
    """A retrieve da aula 8 com uma coluna a mais: cada hit carrega o id do chunk. Nada mais muda."""
    assert col.metadata.get("embedding_model") == EMB_MODEL, "índice e consulta com modelos de embedding diferentes"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"id": i, "text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

def register_rows(rows):
    ROWS_BY_ID.clear()
    ROWS_BY_ID.update({r["id"]: r for r in rows})

def hit_from_row(row, **extra):
    h = {"id": row["id"], "text": row["text"], "doc": row["doc"], "section": row["section"], "distance": float("nan")}
    h.update(extra)
    return h

def dense_ranking(col, text, n=10):
    """Ids dos n chunks mais próximos do texto, na ordem do Chroma (distância crescente)."""
    res = col.query(query_embeddings=embed([text]), n_results=n, include=["distances"])
    return list(zip(res["ids"][0], res["distances"][0]))

def show(hits, score_key="distance"):
    for i, h in enumerate(hits, 1):
        score = h.get(score_key, float("nan"))
        tag = f"({score_key} {score:.3f})" if isinstance(score, float) and not math.isnan(score) else ""
        print(f"[{i}] {h['doc']} · {h['section']}  {tag}")
        print("    " + h["text"][:140].replace("\n", " ") + " ...")

# ===================================================================================================
# Família 1 · HyDE (Hypothetical Document Embeddings)
# A pergunta é curta e coloquial; os chunks são parágrafos formais. Em vez de embutir a pergunta,
# pedimos ao gerador um parágrafo hipotético "no estilo do manual" e embutimos ESSE parágrafo.
# ===================================================================================================
HYDE_SYSTEM = ("You are writing one paragraph of the Aurora Racing team handbook. "
               "Write the paragraph that would answer the question below, in the handbook's formal style, "
               "with concrete values and procedure names. Write only the paragraph. Never say you are unsure.")

def hypothetical_passage(question, model=GEN_MODEL):
    text, _ = generate([{"role": "system", "content": HYDE_SYSTEM}, {"role": "user", "content": question}], model=model)
    return text

def retrieve_hyde(col, question, k=4):
    passage = hypothetical_passage(question)
    hits = [hit_from_row(ROWS_BY_ID[i], distance=d) for i, d in dense_ranking(col, passage, n=k)]
    for h in hits:
        h["hyde_passage"] = passage
    return hits

# ===================================================================================================
# Família 2 · Híbrido BM25 + denso, fundido por RRF (Reciprocal Rank Fusion)
# Vetores acham significado; BM25 acha palavras exatas (códigos como P88-W1, números como 23.5).
# Os dois rankings são fundidos por POSIÇÃO, não por score, porque os scores não são comparáveis.
# ===================================================================================================
def bm25_tokens(text):
    # mantém códigos (p88-w1), números decimais (23.5) e hífens internos; separa o resto
    return re.findall(r"[a-z0-9]+(?:[\-\.][a-z0-9]+)*", text.lower())

class BM25:
    def __init__(self, rows, k1=1.5, b=0.75):
        self.rows, self.k1, self.b = rows, k1, b
        self.docs = [Counter(bm25_tokens(r["text"])) for r in rows]
        self.lens = [sum(c.values()) for c in self.docs]
        self.avg_len = sum(self.lens) / len(self.lens)
        df = Counter()
        for c in self.docs:
            df.update(c.keys())
        N = len(rows)
        self.idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}

    def score(self, query, idx):
        c, L = self.docs[idx], self.lens[idx]
        s = 0.0
        for t in bm25_tokens(query):
            if t not in c:
                continue
            tf = c[t]
            s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * L / self.avg_len))
        return s

    def ranking(self, query, n=10):
        scored = sorted(((self.score(query, i), i) for i in range(len(self.rows))), reverse=True)
        return [(self.rows[i]["id"], s) for s, i in scored[:n] if s > 0]

def rrf(rankings, c=60):
    """rankings: lista de listas de ids (cada uma já ordenada). Devolve [(id, score)] fundido."""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            fused[doc_id] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

BM25_INDEX = None                                   # construído por build_hybrid_index(rows)

def build_hybrid_index(rows, name, **kw):
    """Troca de build_index: o mesmo Chroma de sempre, mais um índice BM25 sobre as mesmas linhas."""
    global BM25_INDEX
    register_rows(rows)
    BM25_INDEX = BM25(rows)
    return build_index(rows, name, **kw)

def retrieve_hybrid(col, question, k=4, n=10, c=60):
    dense = [i for i, _ in dense_ranking(col, question, n=n)]
    sparse = [i for i, _ in BM25_INDEX.ranking(question, n=n)]
    fused = rrf([dense, sparse], c=c)[:k]
    return [hit_from_row(ROWS_BY_ID[i], rrf=s,
                         dense_rank=(dense.index(i) + 1 if i in dense else None),
                         bm25_rank=(sparse.index(i) + 1 if i in sparse else None)) for i, s in fused]

# ===================================================================================================
# Família 3 · Reranking com cross-encoder
# O bi-encoder (embedding) vê pergunta e chunk separados. O cross-encoder lê os dois JUNTOS e devolve
# um score de relevância. É lento demais para varrer o corpus, então recupera-se n candidatos
# baratos e reordena-se só esses. O prompt recebe os k melhores, na ordem do cross-encoder.
# ===================================================================================================
RERANKER = None                                     # carregado por load_reranker()

def load_reranker(name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
    global RERANKER
    from sentence_transformers import CrossEncoder
    RERANKER = CrossEncoder(name)
    return RERANKER

def retrieve_rerank(col, question, k=4, n=12):
    cands = [hit_from_row(ROWS_BY_ID[i], distance=d, dense_rank=r) for r, (i, d) in enumerate(dense_ranking(col, question, n=n), 1)]
    scores = RERANKER.predict([(question, h["text"]) for h in cands])
    for h, s in zip(cands, scores):
        h["rerank"] = float(s)
    return sorted(cands, key=lambda h: -h["rerank"])[:k]

# ===================================================================================================
# Família 4 · CRAG (Corrective RAG)
# A busca devolve SEMPRE k chunks, relevantes ou não. O CRAG põe um avaliador entre a busca e o
# prompt: cada chunk recebe "yes" ou "no". Sem nenhum "yes", a pergunta é reescrita e a busca repete.
# Ainda sem "yes", o prompt declara que não há evidência, e o gerador recusa com base nisso.
# ===================================================================================================
GRADER_SYSTEM = ("You grade passages for the Aurora Racing pit-wall assistant. "
                 "Answer yes if the passage contains any fact that helps answer the question, even partially. "
                 "Answer no only if the passage is about something else. Reply with one word: yes or no.")
REWRITE_SYSTEM = ("Rewrite the question below as a short search query for a motorsport team handbook, "
                  "using the vocabulary a handbook would use. Output only the query.")

def grade_hit(question, hit, model=GEN_MODEL):
    # passagem antes, pergunta repetida no fim: o modelo pequeno responde melhor sobre o que leu por último.
    # "even partially" é o critério certo para um avaliador de contexto; "strict" fazia o 3B recusar pedaços bons.
    msg = [{"role": "system", "content": GRADER_SYSTEM},
           {"role": "user", "content": f"Passage:\n{hit['text']}\n\nQuestion: {question}\n\nDoes the passage help answer the question? Answer yes or no."}]
    verdict, _ = generate(msg, model=model)
    return "yes" in verdict.strip().lower()[:12]

def rewrite_query(question, model=GEN_MODEL):
    text, _ = generate([{"role": "system", "content": REWRITE_SYSTEM}, {"role": "user", "content": question}], model=model)
    return text.strip().strip('"')

def retrieve_crag(col, question, k=4, retriever=retrieve):
    hits = retriever(col, question, k=k)
    for h in hits:
        h["relevant"] = grade_hit(question, h)
    kept = [h for h in hits if h["relevant"]]
    if not kept:                                    # segunda chance: reescreve e busca de novo
        q2 = rewrite_query(question)
        hits2 = retriever(col, q2, k=k)
        for h in hits2:
            h["relevant"] = grade_hit(question, h)
            h["rewritten_query"] = q2
        kept = [h for h in hits2 if h["relevant"]]
    return kept                                     # pode ser vazio, e isso é informação

def build_prompt_crag(question, hits, grounded=True):
    """Troca de build_prompt: sem evidência, o contexto diz isso explicitamente em vez de sumir."""
    if hits:
        return build_prompt(question, hits, grounded=grounded)
    user = ("Context:\n(No relevant passage was found in the handbook for this question.)\n\n"
            f"Question: {question}")
    return [{"role": "system", "content": GROUNDING}, {"role": "user", "content": user}]

# ===================================================================================================
# Harness v2 · a mesma nota da aula 8 (grade), agora com custo em chamadas e latência ponta a ponta
# ===================================================================================================
REFUSAL = "the handbook does not cover this"

def grade(answer, q):
    """(acertou, recusou). Acerto = todas as palavras-chave presentes, ou recusa quando era esperada."""
    a = answer.lower()
    refused = REFUSAL in a
    if q["expect_refusal"]:
        return refused, refused
    return all(kw.lower() in a for kw in q["must_contain"]) and not refused, refused

def run_variant(label, index, questions, retriever, k=4, prompt_builder=build_prompt, model=GEN_MODEL, verbose=True):
    global LLM_CALLS
    records = []
    for q in questions:
        LLM_CALLS = 0
        t0 = time.time()
        hits = retriever(index, q["question"], k=k)
        answer, _ = generate(prompt_builder(q["question"], hits), model=model)
        total = time.time() - t0
        correct, refused = grade(answer, q)
        records.append({"label": label, "model": model, "k": k, "id": q["id"], "tier": q["tier"],
                        "question": q["question"], "retrieved_docs": "|".join(h["doc"] for h in hits),
                        "n_hits": len(hits), "gold_hit": any(h["doc"] in q["gold_docs"] for h in hits),
                        "gold_rank": next((i for i, h in enumerate(hits, 1) if h["doc"] in q["gold_docs"]), None),
                        "answer": answer, "correct": correct, "refused": refused,
                        "llm_calls": LLM_CALLS, "latency_s": round(total, 2)})
        if verbose:
            print(f"{label:<9} {q['id']} {'✓' if correct else '✗'}  {LLM_CALLS} calls  {total:4.1f}s")
    return pd.DataFrame(records)

def summarize(dfs):
    """Uma linha por variante: acerto por tier, gold_hit nas respondíveis, chamadas e latência médias."""
    df = pd.concat(dfs, ignore_index=True)
    acc = df.pivot_table(index="label", columns="tier", values="correct", aggfunc="mean")
    ans = df[df["tier"] != "none"]
    extra = df.groupby("label").agg(overall=("correct", "mean"), llm_calls=("llm_calls", "mean"), latency_s=("latency_s", "mean"))
    retr = ans.groupby("label").agg(gold_hit=("gold_hit", "mean"), gold_at_1=("gold_rank", lambda s: (s == 1).mean()))
    out = acc.join(extra).join(retr)[["direct", "multi", "none", "overall", "gold_hit", "gold_at_1", "llm_calls", "latency_s"]].round(2)
    order = [l for l in ["baseline", "hyde", "hybrid", "rerank", "crag"] if l in out.index]
    return out.loc[order + [l for l in out.index if l not in order]]


In [ ]:
# Gráfico final. A baseline é o fundo (cinza); as variantes são a figura, uma cor por família.
import matplotlib.pyplot as plt
import numpy as np

FAMILY_COLOR = {"baseline": "#9A9A9A", "hyde": "#5B7DB1", "hybrid": "#2E5E4E", "rerank": "#C98A2B", "crag": "#A63D40"}
FAMILY_NAME  = {"baseline": "Classic RAG (baseline)", "hyde": "F1 · HyDE (the question)", "hybrid": "F2 · Hybrid + RRF (the search)",
                "rerank": "F3 · Reranking (the context)", "crag": "F4 · CRAG (the control loop)"}

def plot_comparison(summary, title="Same corpus, same 20 questions, same grader"):
    tiers = ["direct", "multi", "none"]
    labels = list(summary.index)
    x = np.arange(len(tiers)); w = 0.8 / len(labels)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [3, 1.2]})
    for j, lab in enumerate(labels):
        vals = [summary.loc[lab, t] for t in tiers]
        ax.bar(x + (j - (len(labels) - 1) / 2) * w, vals, w, color=FAMILY_COLOR.get(lab, "#666"), label=FAMILY_NAME.get(lab, lab),
               alpha=1.0 if lab != "baseline" else 0.9)
    ax.set_xticks(x); ax.set_xticklabels(["direct (8)", "multi (6)", "unanswerable (6)"]); ax.set_ylim(0, 1.05)
    ax.set_ylabel("accuracy"); ax.set_title(title, loc="left"); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
    ax2.barh(labels[::-1], [summary.loc[l, "llm_calls"] for l in labels[::-1]], color=[FAMILY_COLOR.get(l, "#666") for l in labels[::-1]])
    ax2.set_xlabel("LLM calls per question"); ax2.set_title("What each fix costs", loc="left"); ax2.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()


---
# Bloco 0 · A linha de base revisitada

**Antes de consertar, diga o que quebrou. Cada linha errada da baseline cai numa família, e a família diz qual peça trocar.**

Você trouxe `results_baseline.csv` da aula 8. Aqui ele é rerodado com o harness desta aula (`run_variant`), para que a comparação com as variantes seja na mesma máquina, no mesmo dia, com a mesma nota. O harness novo tem duas colunas a mais: `llm_calls` (quantas vezes o gerador foi chamado por pergunta) e `gold_rank` (em que posição o primeiro chunk-ouro apareceu, ou vazio se não apareceu).

In [ ]:
docs      = load_corpus()
questions = json.load(open("questions.json", encoding="utf-8"))
rows      = make_chunks(docs, chunk_size=512, overlap=51)
index = build_hybrid_index(rows, "handbook_512")      # o Chroma de sempre mais um BM25 (usado a partir do Bloco 2)
warm_up()

df_base = run_variant("baseline", index, questions, retrieve, k=4)
df_base.to_csv("results_baseline_v2.csv", index=False)
df_base.groupby("tier")["correct"].mean().round(2)

**O que observar.** Uma coluna de acertos por tier. Se a sua aula 8 foi típica, as diretas ficam perto de 1.0, as compostas entre 0.3 e 0.7 e as sem resposta entre 0.5 e 0.8 (o modelo às vezes inventa mesmo com a regra de grounding). Cada linha custou exatamente 1 chamada ao LLM; guarde esse número, ele é o denominador de tudo o que vem.

### Complete o código
Escreva `box(row)` que classifica uma linha do harness em `"ok"`, `"retrieval"` (não acertou e o chunk-ouro não veio), `"generation"` (não acertou, o chunk-ouro veio) ou `"refusal"` (tier `none` e o modelo não recusou). Depois conte as caixas.

In [ ]:
def box(row) -> str:
    # ---- SEU CÓDIGO AQUI ----
    # ordem das regras: ok · refusal (tier none) · retrieval (gold_hit False) · generation
    ...

df_base["box"] = df_base.apply(box, axis=1)
df_base.groupby(["tier", "box"]).size().unstack(fill_value=0)

**Como saber se deu certo.** A soma das quatro caixas é 20. Nenhuma linha `none` pode cair em `retrieval` ou `generation` (as sem resposta não têm chunk-ouro). Toda linha `ok` tem `correct == True`.

<details>
<summary><b>Solução de referência</b></summary>

```python
def box(row) -> str:
    if row["correct"]:
        return "ok"
    if row["tier"] == "none":
        return "refusal"
    return "retrieval" if not row["gold_hit"] else "generation"
```
</details>

As caixas mapeiam nas famílias: `retrieval` pede F1 ou F2 (a busca não achou), `generation` pede F3 (achou, mas o modelo não leu direito, muitas vezes por ordem ou excesso de contexto) e `refusal` pede F4 (o modelo confiou num chunk que não devia).

---
# Bloco 1 · F1 · HyDE · a pergunta vira parágrafo antes de virar vetor

**Um embedding compara texto com texto. Uma pergunta curta e um parágrafo formal são textos diferentes mesmo quando falam da mesma coisa. HyDE pede ao gerador um parágrafo hipotético e embute o parágrafo.**

O parágrafo é inventado. Os números estão errados, os nomes podem estar errados. Não importa: o que a busca usa é o formato e o vocabulário, e nisso o gerador acerta. É a troca de `retrieve` na entrada. O prompt final continua recebendo os chunks verdadeiros do manual, não o parágrafo.

In [ ]:
for Q in ["How much air do we put in the tyres of car 27 when it rains?",
          "How many championship points does car 27 have this season?"]:
    print("=" * 100); print(Q)
    base = retrieve(index, Q, k=3)
    hyde = retrieve_hyde(index, Q, k=3)
    print("\n-- parágrafo hipotético --\n" + hyde[0]["hyde_passage"])
    print("\n-- baseline (embute a pergunta) --"); show(base)
    print("\n-- HyDE (embute o parágrafo) --"); show(hyde)

**O que observar.** Para a primeira pergunta, as distâncias do HyDE ficam menores que as da baseline (parágrafo se parece com parágrafo), e o primeiro chunk é o mesmo ou melhor (seção 4.2 ou tabela de pressões). Para a `n01`, o gerador escreve com toda a confiança um parágrafo sobre pontos no campeonato, e a busca devolve os chunks mais parecidos com esse parágrafo inventado. Nada no manual fala de pontos, então os chunks vêm de qualquer lugar, com distâncias razoáveis. **HyDE melhora a pergunta, não a coragem de recusar.** Guarde isso para o Bloco 4.

### Complete o código
O artigo original do HyDE não descarta a pergunta: embute a pergunta e o parágrafo e usa a **média** dos dois vetores. Escreva `retrieve_hyde_avg(col, question, k)` com essa média. Depois compare os três (`retrieve`, `retrieve_hyde`, `retrieve_hyde_avg`) em `gold_at_1` sobre as 14 perguntas com resposta, sem chamar o gerador final (só recuperação).

In [ ]:
import numpy as np

def retrieve_hyde_avg(col, question, k=4):
    passage = hypothetical_passage(question)
    # ---- SEU CÓDIGO AQUI ----
    # vq, vp = embed([question, passage]); v = média dos dois; consulte col.query(query_embeddings=[v], n_results=k, include=["distances"])
    # e devolva [hit_from_row(ROWS_BY_ID[i], distance=d) for i, d in zip(res["ids"][0], res["distances"][0])]
    ...

answerable = [q for q in questions if not q["expect_refusal"]]

def gold_at_1(retriever, qs, k=4) -> float:
    return sum(retriever(index, q["question"], k=k)[0]["doc"] in q["gold_docs"] for q in qs) / len(qs)

for name, fn in [("baseline", retrieve), ("hyde", retrieve_hyde), ("hyde_avg", retrieve_hyde_avg)]:
    print(f"{name:<10} gold@1 = {gold_at_1(fn, answerable):.2f}")

**Como saber se deu certo.** Os três números ficam entre 0.6 e 1.0. Com este corpus de 14 chunks a baseline já é forte, então a diferença é pequena e pode ir para qualquer lado. O que importa é que `hyde_avg` nunca deve ficar muito abaixo dos outros dois; se ficar, verifique se os dois vetores foram normalizados antes da média (o Chroma usa cosseno, então a escala não importa, mas a média de um vetor grande com um pequeno pesa para o grande).

<details>
<summary><b>Solução de referência</b></summary>

```python
def retrieve_hyde_avg(col, question, k=4):
    passage = hypothetical_passage(question)
    vq, vp = (np.asarray(v) for v in embed([question, passage]))
    vq, vp = vq / np.linalg.norm(vq), vp / np.linalg.norm(vp)
    v = ((vq + vp) / 2).tolist()
    res = col.query(query_embeddings=[v], n_results=k, include=["distances"])
    return [hit_from_row(ROWS_BY_ID[i], distance=d) for i, d in zip(res["ids"][0], res["distances"][0])]
```
</details>

In [ ]:
df_hyde = run_variant("hyde", index, questions, retrieve_hyde, k=4)
df_hyde.to_csv("results_hyde.csv", index=False)
summarize([df_base, df_hyde])

---
# Bloco 2 · F2 · Híbrido · dois rankings fundidos por posição

**Vetores acham significado; BM25 acha palavras exatas. Códigos, números e nomes raros são palavras exatas. O híbrido roda os dois e funde por posição (RRF), porque os scores não são comparáveis.**

BM25 é o algoritmo de busca lexical dos mecanismos de busca clássicos: conta quantas vezes cada termo da pergunta aparece no chunk, dá mais peso a termos raros no corpus (`idf`) e satura a contagem (`k1`) e o tamanho do chunk (`b`). A classe `BM25` no núcleo tem 25 linhas; leia-a. É a troca de `build_index` (dois índices) e de `retrieve` (uma fusão).

In [ ]:
Q = "Which procedure code names the wet-race pit stop for car 88?"
print("tokens BM25 da pergunta:", bm25_tokens(Q))
print("idf de alguns termos    :", {t: round(BM25_INDEX.idf.get(t, 0), 2) for t in ["pit", "stop", "wet-race", "p88-w1", "car"]})

dense  = dense_ranking(index, Q, n=6)
sparse = BM25_INDEX.ranking(Q, n=6)
print("\ndenso :", [f"{i} ({d:.2f})" for i, d in dense])
print("BM25  :", [f"{i} ({s:.1f})" for i, s in sparse])
print()
for h in retrieve_hybrid(index, Q, k=4):
    print(f"rrf {h['rrf']:.4f}  denso #{h['dense_rank']}  bm25 #{h['bm25_rank']}   {h['id']} · {h['section'][:55]}")

**O que observar.** `p88-w1` tem o maior `idf` da lista (aparece em poucos chunks), e `car` um dos menores. No BM25 o chunk de `procedure_codes.md` ou o da seção 4.2 vem em primeiro com folga. No denso ele costuma vir também, mas pode empatar com chunks que falam de "pit stop" sem o código. A fusão põe em primeiro quem está bem nos dois rankings; um chunk que está em primeiro no BM25 e ausente no denso perde para um que está em segundo nos dois. É essa a lógica do RRF: recompensa consenso.

### Complete o código
O núcleo já traz `rrf`. Escreva a sua própria versão, `my_rrf(rankings, c=60)`, sem olhar, e confira com o exemplo abaixo. Depois responda em uma linha: por que `c=60` e não `c=0`?

In [ ]:
from collections import defaultdict

def my_rrf(rankings: list[list[str]], c: int = 60) -> list[tuple[str, float]]:
    # ---- SEU CÓDIGO AQUI ----
    # para cada ranking, para cada (posição a partir de 1, id): score[id] += 1 / (c + posição). Devolva ordenado por score decrescente.
    ...

print(my_rrf([["a", "b", "c"], ["c", "a", "d"]]))
print(rrf([["a", "b", "c"], ["c", "a", "d"]]))

**Como saber se deu certo.** As duas listas são iguais: `a` primeiro (1º e 2º), `c` segundo (3º e 1º), depois `b` e `d`. Com `c=0` a primeira posição valeria 1.0 e a segunda 0.5, e um único primeiro lugar dominaria tudo; `c=60` achata a curva para que estar em 2º e 3º valha mais que estar em 1º e ausente.

<details>
<summary><b>Solução de referência</b></summary>

```python
def my_rrf(rankings, c=60):
    score = defaultdict(float)
    for ranking in rankings:
        for pos, doc_id in enumerate(ranking, 1):
            score[doc_id] += 1 / (c + pos)
    return sorted(score.items(), key=lambda x: -x[1])
```
</details>

In [ ]:
df_hybrid = run_variant("hybrid", index, questions, retrieve_hybrid, k=4)
df_hybrid.to_csv("results_hybrid.csv", index=False)
summarize([df_base, df_hyde, df_hybrid])

---
# Bloco 3 · F3 · Reranking · recupera muitos baratos, reordena poucos com um leitor caro

**O embedding vê pergunta e chunk separados (bi-encoder). Um cross-encoder lê os dois juntos e devolve um score de relevância. É lento demais para o corpus inteiro, então roda só sobre os candidatos que o embedding trouxe.**

É a troca de `retrieve` na saída: `n=12` candidatos entram, o cross-encoder pontua cada par (pergunta, chunk), os `k=4` melhores seguem para o prompt na ordem do cross-encoder. Isso ataca dois problemas de uma vez: o chunk certo em 6º lugar (não entraria) e o chunk certo no meio do prompt (*lost in the middle*).

In [ ]:
load_reranker()                                   # cross-encoder/ms-marco-MiniLM-L-6-v2, ~90 MB, roda em CPU

Q = "Why does car 88 use a higher rear tyre pressure than car 27 in the wet?"
hits = retrieve_rerank(index, Q, k=4, n=12)
print(f"{'final':>5} {'denso':>5} {'rerank':>7}  chunk")
for i, h in enumerate(hits, 1):
    print(f"{i:>5} {h['dense_rank']:>5} {h['rerank']:>7.2f}  {h['id']} · {h['section'][:55]}")
answer, secs = generate(build_prompt(Q, hits))
print(f"\n{answer}")

**O que observar.** Os scores do cross-encoder são logits (podem ser negativos); só a ordem importa. O chunk da seção 4.2 e o do `drivers.md` (Mika Sørensen) devem estar entre os 4 finais, e a resposta deve mencionar a barra anti-rolagem (*anti-roll bar*) citando os dois arquivos. Compare `final` com `denso`: se nada mudou de posição, o embedding já tinha acertado a ordem; troque a pergunta por `m06` e veja se o padrão se repete.

### Complete o código
Reordenar por relevância resolve o que entra. *Lost in the middle* é sobre onde fica. Escreva `sandwich(hits)` que recebe os chunks já ordenados por relevância e devolve uma lista em que o 1º fica no início, o 2º no fim, o 3º logo após o 1º, o 4º logo antes do 2º, e assim por diante (os melhores nas bordas, os piores no meio).

In [ ]:
def sandwich(hits: list) -> list:
    # ---- SEU CÓDIGO AQUI ----
    # dica: posições ímpares (1º, 3º, 5º...) vão para a frente na ordem; posições pares vão para o fim em ordem invertida
    ...

print([h for h in sandwich(list("abcdef"))])      # funciona com qualquer lista; aqui letras no lugar de chunks

def build_prompt_sandwich(question, hits, grounded=True):
    return build_prompt(question, sandwich(hits), grounded=grounded)

**Como saber se deu certo.** `sandwich(list("abcdef"))` deve devolver `['a', 'c', 'e', 'f', 'd', 'b']`. Com 4 chunks: `['a', 'c', 'd', 'b']`.

<details>
<summary><b>Solução de referência</b></summary>

```python
def sandwich(hits):
    front = hits[0::2]            # 1º, 3º, 5º ...
    back  = hits[1::2][::-1]      # 2º, 4º, 6º ... invertidos
    return front + back
```
</details>

Com `k=4` o efeito do sanduíche é pequeno (o meio tem só dois chunks). Ele importa quando `k` cresce para 10 ou 20, que é exatamente quando o reranking permite crescer `k` sem afogar o modelo.

In [ ]:
df_rerank = run_variant("rerank", index, questions, retrieve_rerank, k=4)
df_rerank.to_csv("results_rerank.csv", index=False)
summarize([df_base, df_hyde, df_hybrid, df_rerank])

---
# Bloco 4 · F4 · CRAG · um avaliador entre a busca e o prompt

**A busca devolve sempre k chunks, relevantes ou não. O RAG clássico confia em todos. O CRAG põe um avaliador no meio: cada chunk recebe sim ou não, e sem nenhum sim o sistema reescreve a pergunta, tenta de novo e, se ainda não houver evidência, diz isso ao gerador em vez de esconder.**

É um laço em volta de `retrieve` mais uma troca de `build_prompt`: quando a lista aprovada vem vazia, o contexto declara "nenhuma passagem relevante" em vez de simplesmente não existir. O gerador recusa com base numa informação, não numa ausência. O CRAG original usa busca na web como segunda chance; aqui a segunda chance é a reescrita da pergunta, que cabe no corpus fechado.

In [ ]:
for Q in ["Who is the team principal of Aurora Racing?",           # sem resposta, com armadilha
          "Who is the race engineer of Luca Ferrand?"]:          # com resposta
    LLM_CALLS = 0
    print("=" * 100); print(Q)
    raw = retrieve(index, Q, k=4)
    for h in raw:
        print(f"  {'yes' if grade_hit(Q, h) else 'no ':<4} {h['id']} · {h['section'][:60]}  (distance {h['distance']:.3f})")
    hits = retrieve_crag(index, Q, k=4)
    if hits and "rewritten_query" in hits[0]:
        print(f"  pergunta reescrita: {hits[0]['rewritten_query']}")
    elif not hits:
        print(f"  pergunta reescrita: {rewrite_query(Q)}   (nenhum chunk aprovado nem depois da reescrita)")
    answer, _ = generate(build_prompt_crag(Q, hits))
    print(f"  aprovados: {[h['id'] for h in hits]}")
    print(f"  resposta : {answer.splitlines()[0][:120]}")
    print(f"  {LLM_CALLS} chamadas ao LLM")

**O que observar.** Para o *team principal*, quatro `no` (o chunk do protocolo fala do cargo, não da pessoa), uma reescrita, mais quatro `no`, e a recusa. Umas 10 a 14 chamadas ao LLM para uma pergunta que a baseline responde com 1 (às vezes inventando um nome). Para o Luca, um ou dois `yes` no chunk do `drivers.md`, e a resposta cita Priya Nandakumar com 5 chamadas. **Esse é o preço da família 4: pagar em chamadas para não pagar em confiança.**

Se o avaliador aprovar o chunk do protocolo para o *team principal*, você acabou de ver o avaliador errar. Ele é um LLM de 3B respondendo sim ou não; é melhor que nada e pior que um humano.

### Complete o código
O avaliador custa tanto quanto o gerador. Um avaliador de 0.5B seria 4 a 5 vezes mais rápido. Escreva `grader_agreement(qs, k)` que, para cada pergunta e cada um dos k chunks recuperados, pergunta ao `qwen2.5:3b` e ao `qwen2.5:0.5b` e devolve a fração de vereditos iguais.

In [ ]:
def grader_agreement(qs: list, k: int = 4) -> float:
    agree = total = 0
    for q in qs:
        for h in retrieve(index, q["question"], k=k):
            # ---- SEU CÓDIGO AQUI ----
            # big = grade_hit(q["question"], h, model="qwen2.5:3b"); small = grade_hit(q["question"], h, model="qwen2.5:0.5b")
            ...
    return agree / total

print(f"concordância 3B vs 0.5B: {grader_agreement(questions[:6] + questions[14:17]):.2f}")   # 9 perguntas para não esperar demais

**Como saber se deu certo.** Um número entre 0.6 e 0.9. Abaixo de 0.5 o avaliador pequeno está próximo de uma moeda; confira se ele está respondendo mesmo "yes"/"no" (imprima os vereditos crus) ou se está escrevendo uma frase que começa de outro jeito.

<details>
<summary><b>Solução de referência</b></summary>

```python
def grader_agreement(qs, k=4):
    agree = total = 0
    for q in qs:
        for h in retrieve(index, q["question"], k=k):
            big   = grade_hit(q["question"], h, model="qwen2.5:3b")
            small = grade_hit(q["question"], h, model="qwen2.5:0.5b")
            agree += (big == small); total += 1
    return agree / total
```
</details>

Se a concordância for alta, `retrieve_crag` pode usar o modelo pequeno como avaliador e o grande só como gerador. É a mesma ideia do reranking: um leitor barato para filtrar, um leitor caro para escrever.

In [ ]:
df_crag = run_variant("crag", index, questions, retrieve_crag, k=4, prompt_builder=build_prompt_crag)
df_crag.to_csv("results_crag.csv", index=False)
summarize([df_base, df_hyde, df_hybrid, df_rerank, df_crag])

---
# Bloco 5 · A tabela · qual conserto pagou o que custou?

**Cinco linhas, o mesmo harness. A pergunta não é "qual é a melhor", é "qual ganho justifica qual custo, neste corpus".**

A baseline é o fundo cinza. Cada família tem uma cor. O painel da direita é o preço em chamadas ao LLM por pergunta. Um corpus de 12 documentos é pequeno e limpo; a baseline já é forte, então alguns consertos vão aparecer como empate. Isso não é um defeito do experimento; é a resposta. Um conserto que empata aqui pode ser decisivo num corpus de 10.000 documentos com códigos e nomes raros.

In [ ]:
ALL = [df_base, df_hyde, df_hybrid, df_rerank, df_crag]
summary = summarize(ALL)
display(summary)
plot_comparison(summary)

**O que observar.** O padrão típico neste corpus (o seu pode variar): as diretas já estão perto de 1.0 e quase ninguém as move; o híbrido tende a ganhar nas compostas, onde códigos e nomes exatos decidem; o reranking sobe `gold_at_1` mesmo quando o acerto não muda; o CRAG protege a coluna `none` e é o mais caro. Leia a coluna `llm_calls` ao lado de `none`: é a troca que a família 4 faz.

### Complete o código
Escreva `cost_benefit(summary)` que devolve, para cada variante, o ganho de acerto geral em relação à baseline (`overall - overall_baseline`) e o custo em chamadas extras por pergunta (`llm_calls - llm_calls_baseline`), ordenado do melhor ganho por chamada para o pior. Variantes sem chamada extra ficam no topo se o ganho for positivo.

In [ ]:
def cost_benefit(summary: pd.DataFrame) -> pd.DataFrame:
    base = summary.loc["baseline"]
    rows_out = []
    for label, r in summary.drop("baseline").iterrows():
        # ---- SEU CÓDIGO AQUI ----
        # gain = r["overall"] - base["overall"]; extra = r["llm_calls"] - base["llm_calls"]; ratio = gain / extra se extra > 0 senão gain
        ...
    return pd.DataFrame(rows_out).sort_values("ratio", ascending=False)

cost_benefit(summary)

**Como saber se deu certo.** Quatro linhas (`hyde`, `hybrid`, `rerank`, `crag`), colunas `gain`, `extra_calls`, `ratio`. `hybrid` e `rerank` têm `extra_calls == 0`. Se todas as `gain` forem zero, os cinco harness deram o mesmo acerto; confira se `summary` tem mesmo cinco linhas diferentes.

<details>
<summary><b>Solução de referência</b></summary>

```python
def cost_benefit(summary):
    base = summary.loc["baseline"]
    rows_out = []
    for label, r in summary.drop("baseline").iterrows():
        gain, extra = r["overall"] - base["overall"], r["llm_calls"] - base["llm_calls"]
        rows_out.append({"variant": label, "gain": round(gain, 2), "extra_calls": round(extra, 1),
                         "ratio": round(gain / extra, 3) if extra > 0 else round(gain, 3)})
    return pd.DataFrame(rows_out).sort_values("ratio", ascending=False)
```
</details>

---
# Exercício da semana · Os botões que a aula deixou fixos

Nenhuma técnica nova. As quatro variantes desta aula rodaram com um parâmetro fixo cada, escolhido por mim. O exercício é mexer nesses botões, medir no mesmo harness e explicar o que mudou e em quais perguntas. As técnicas só citadas (Step-Back, multi-query, parent-child, Adaptive RAG) ficam para o trabalho da unidade.

> **Qual botão mudou o placar, e o que isso diz sobre a técnica?**

### Os botões (escolha DUAS famílias)

| Família | Botão | Como girar | O que observar |
|---|---|---|---|
| F1 · HyDE | quantas hipotéticas e o que entra na média | escreva `retrieve_hyde3`: três `hypothetical_passage` com `generate(..., temperature=0.7)`, média dos três vetores com o da pergunta; ou embuta só a hipotética, sem a pergunta | `gold_hit` e a faixa `none` |
| F2 · Híbrido | a constante do RRF e o tamanho do pool | `retrieve_hybrid(index, q, k=4, c=10)`, `c=200`, e `n=20` em vez de 10 | a ordem dos quatro na pergunta do P27-W1 e a faixa `multi` |
| F3 · Reranking | o número de candidatos e um limiar | `retrieve_rerank(..., n=8)`, `n=20`; e um `retrieve_rerank_cut` que descarta chunks com `rerank < 0` (o chunk de garagem do Bloco 3 tinha −0.98) | a faixa `none`: descartar negativos muda a recusa? |
| F4 · CRAG | o prompt do avaliador | volte ao prompt "strict" (está no comentário de `grade_hit`) e compare com o atual; ou faça `retrieve_crag(..., retriever=retrieve_hybrid)` | `llm_calls`, e quais chunks bons o avaliador descarta |

Cada botão é uma função nova de poucas linhas que **chama** as do núcleo com outro argumento, ou uma cópia da função com uma linha trocada. Nada de reescrever a técnica.

### Construir

1. Para cada uma das duas famílias escolhidas, uma variante com o botão girado, `run_variant` sobre as 20 perguntas, `results_<nome>.csv`.
2. A tabela do Bloco 5 com **sete linhas** (as cinco da aula mais as suas duas) e o `cost_benefit`.

### Analisar

3. Para cada variante sua, a lista das perguntas em que ela e a versão da aula **discordam** (uma acertou, a outra não), com uma frase por pergunta dizendo por quê. Olhe `retrieved_docs` e `answer` no CSV antes de escrever.
4. Um parágrafo por família respondendo à pergunta em negrito. Se o botão não mudou nada, diga por que não mudou neste corpus e em que corpus mudaria.
5. Rode a baseline **três vezes** (`run_variant("baseline_r1", ...)`, `r2`, `r3`) e liste as perguntas cujo resultado muda entre rodadas. Use isso para dizer, na sua tabela, quais diferenças são maiores que o ruído.

### Entregar

Um relatório de **duas páginas** (PDF ou notebook exportado) com a tabela de sete linhas, o `cost_benefit`, as duas listas de discordância, os dois parágrafos e a lista de perguntas instáveis do item 5. Anexe os CSVs.

### Bônus (opcional)

A variante empilhada, `hybrid+rerank+crag` (a toca do coelho do Bloco 5 diz como), como oitava linha. Os ganhos somam, ou o CRAG desfaz o que o híbrido ganhou?


---
# Tempos das células

Um retrato de quanto custou cada passo nesta máquina. Compare o harness da baseline (20 chamadas) com o do CRAG (100 ou mais), e a carga do cross-encoder com uma resposta do gerador. Anexe esta tabela ao relatório do exercício.

In [ ]:
# Resumo dos tempos de todas as células rodadas nesta sessão.
import pandas as pd
times = pd.DataFrame(CELL_TIMES)
print(f"total: {times['seconds'].sum():.1f} s em {len(times)} células")
times
